<a href="https://colab.research.google.com/github/Ayushi-369/FlyrankAi-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayushi-369/FlyrankAi-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One row = One published page (URL).

Time Window: Evaluated over a trailing 90-day search history window.

Target Label: needs_refresh (1 = Page has severe traffic decay, 0 = Page is healthy).

Excluded Field: impressions_next_30d (Excluded because future traffic occurs after the decision moment).

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd

# Load dataset safely
path = "../../data/raw/content_refresh_anonymized.csv" if os.path.exists("../../data/raw/content_refresh_anonymized.csv") else "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(path)

target_col = "needs_refresh" if "needs_refresh" in df.columns else "target"

# Check grain (uniqueness)
print(f"Total Rows in Dataset: {len(df):,}")
print(f"Is page_id unique per row? {df['page_id'].is_unique if 'page_id' in df.columns else True}")

Total Rows in Dataset: 30,000
Is page_id unique per row? True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: content_age_days, days_since_last_update, avg_position, ctr, engagement_rate

Label: needs_refresh

Context Fields: page_id, content_type, position_tier

Excluded: impressions_next_30d (Excluded to prevent using future data to predict the past).

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show available feature columns
feature_cols = ["content_age_days", "days_since_last_update", "avg_position", "ctr", "engagement_rate"]
available_cols = [c for c in feature_cols if c in df.columns]
print("Selected Features available in dataframe:")
print(df[available_cols].head(3))

Selected Features available in dataframe:
   content_age_days  days_since_last_update  avg_position   ctr  \
0               187                      20          10.6  0.76   
1               445                      25          20.3  0.05   
2               141                      20          36.5  0.09   

   engagement_rate  
0             5.88  
1             0.00  
2             0.00  


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verified row count, feature availability, built a 5-feature frame, and tested feature leakage by intentionally injecting a future target label.

In [10]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score

# Safely load df if it's not present in the current namespace
if 'df' not in locals() and 'df' not in globals():
    print("Variable 'df' not found in active namespace. Loading dataset...")
    path = "../../data/raw/content_refresh_anonymized.csv" if os.path.exists("../../data/raw/content_refresh_anonymized.csv") else "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    df = pd.read_csv(path)

# 1. Check Row Count & Availability (Surviving Filter)
survived_rows = df[(df["content_age_days"].notna()) & (df["avg_position"] > 0) & (df["ctr"].notna())].copy()
print(f"Rows surviving valid feature check: {len(survived_rows):,} / {len(df):,}")

# 2. Programmatically define our target label: 'needs_refresh'
# We define a page as needing a refresh (1) if its clicks in the last 30 days
# dropped by more than 50% compared to the previous 30 days, and it has some historical clicks.
clicks_last = survived_rows["clicks_last_30d"]
clicks_prev = survived_rows["clicks_prev_30d"]
survived_rows["needs_refresh"] = ((clicks_last < 0.5 * clicks_prev) & (clicks_prev > 0)).astype(int)

current_target_col = "needs_refresh"
print(f"Created and verified target column: '{current_target_col}' ({survived_rows[current_target_col].sum()} positive cases)")

# 3. Build 5-Feature Frame
features = ["content_age_days", "days_since_last_update", "avg_position", "ctr", "engagement_rate"]
X = survived_rows[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = survived_rows[current_target_col]

# 4. Leakage Trap Experiment
# Step A: Add a leaked column derived directly from the target
survived_rows_copy = survived_rows.copy()
survived_rows_copy["leaked_future_flag"] = y

# Score with leaked feature
leaky_pred = (survived_rows_copy["leaked_future_flag"] > 0.5).astype(int)
print(f"⚠️ Precision WITH Leaked Column (Artificially Perfect): {precision_score(y, leaky_pred):.4f}")

# Step B: Remove leaked column and report honest baseline
survived_rows_copy.drop(columns=["leaked_future_flag"], inplace=True)
honest_pred = (survived_rows_copy["avg_position"] > 20).astype(int)
print(f"✅ Precision AFTER Deleting Leaked Column (Honest Baseline): {precision_score(y, honest_pred):.4f}")

Rows surviving valid feature check: 28,795 / 30,000
Created and verified target column: 'needs_refresh' (3936 positive cases)
⚠️ Precision WITH Leaked Column (Artificially Perfect): 1.0000
✅ Precision AFTER Deleting Leaked Column (Honest Baseline): 0.1116


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced History: Newer articles do not have the same 90-day history depth as older published articles.

External Search Factors: The data records impressions and CTR, but cannot observe external factors like sudden competitor ranking changes or seasonal search volume dips

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Data limit check completed: No external competitor metrics present in dataset.")

Data limit check completed: No external competitor metrics present in dataset.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.